# 

In [5]:
import pandas as pd
import numpy as np
column_names = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
               'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
               'wheel-base', 'length', 'width', 'height', 'curb-weight',
               'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system',
               'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm',
               'city-mpg', 'highway-mpg', 'price']

# Load data, recognizing '?' as NaN
data = pd.read_csv('datasets/automobile/automobile/imports-85.data', header=None, names=column_names, sep=',', na_values='?')

print("Automobile data loaded successfully!")
print("Initial shape:", data.shape)

Automobile data loaded successfully!
Initial shape: (205, 26)


In [6]:
print("\nFirst 5 rows:")
print(data.head())

print("\nData Info:")
print(data.info())

print("\nNumerical Summary:")

print(data.describe(include='all'))

print("\nMissing Values per Column:")
print(data.isnull().sum())

print("\nValue Counts for 'body-style':")
print(data['body-style'].value_counts())

target_variable = 'price'
print(f"\nTarget Variable: {target_variable}")


First 5 rows:
   symboling  normalized-losses         make fuel-type aspiration  \
0          3                NaN  alfa-romero       gas        std   
1          3                NaN  alfa-romero       gas        std   
2          1                NaN  alfa-romero       gas        std   
3          2              164.0         audi       gas        std   
4          2              164.0         audi       gas        std   

  num-of-doors   body-style drive-wheels engine-location  wheel-base  ...  \
0          two  convertible          rwd           front        88.6  ...   
1          two  convertible          rwd           front        88.6  ...   
2          two    hatchback          rwd           front        94.5  ...   
3         four        sedan          fwd           front        99.8  ...   
4         four        sedan          4wd           front        99.4  ...   

   engine-size  fuel-system  bore  stroke compression-ratio horsepower  \
0          130         mpfi  3.47

In [7]:
data.dropna(subset=[target_variable], axis=0, inplace=True)
print(f"\nDropped rows where target '{target_variable}' was missing. New shape: {data.shape}")

# 2. Impute numerical features with the mean
numerical_cols_to_impute = ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm']
for col in numerical_cols_to_impute:
    # Ensure column is numeric before imputing
    data[col] = pd.to_numeric(data[col], errors='coerce') # Should already be numeric if loaded correctly, but safe check
    mean_val = data[col].mean()
    data[col].fillna(mean_val, inplace=True)
    print(f"Filled NaNs in numerical '{col}' with mean: {mean_val:.2f}")

# 3. Impute categorical features with the mode
categorical_cols_to_impute = ['num-of-doors']
for col in categorical_cols_to_impute:
    mode_val = data[col].mode()[0]
    data[col].fillna(mode_val, inplace=True)
    print(f"Filled NaNs in categorical '{col}' with mode: {mode_val}")

# Verify no more missing values (except maybe ones you decided not to handle)
print("\nMissing Values after Imputation:")
print(data.isnull().sum()) # Should show 0 for imputed columns


# B) Convert Categorical Features to Numerical (One-Hot Encoding)
# Identify remaining categorical columns (object type)
categorical_cols = data.select_dtypes(include='object').columns
print(f"\nCategorical columns to encode: {list(categorical_cols)}")

# Apply one-hot encoding
data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

print("\nData after one-hot encoding (first 5 rows):")
print(data_encoded.head())
print("\nNew shape after encoding:", data_encoded.shape)


# C) Ensure Target is Numeric and Separate Features/Target
# Convert target 'price' to numeric (it might still be object if loaded with '?')
data_encoded[target_variable] = pd.to_numeric(data_encoded[target_variable], errors='coerce')

# Separate Features (X) and Target (y)
X = data_encoded.drop(target_variable, axis=1)
y = data_encoded[target_variable]

# Verify separation
print(f"\nFeatures (X) and Target (y) separated.")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Data type of y:", y.dtype) # Should be numeric now


# D) Feature Scaling (To be done within Pipeline)
# Identify numerical feature columns in the final feature set X for the scaler
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
print(f"\nNumerical features identified for scaling: {len(numerical_features)}")

# --- End of Pre-processing ---
# 'X' contains cleaned features (numerical + one-hot encoded)
# 'y' contains the cleaned numerical target ('price')


Dropped rows where target 'price' was missing. New shape: (201, 26)
Filled NaNs in numerical 'normalized-losses' with mean: 122.00
Filled NaNs in numerical 'bore' with mean: 3.33
Filled NaNs in numerical 'stroke' with mean: 3.26
Filled NaNs in numerical 'horsepower' with mean: 103.40
Filled NaNs in numerical 'peak-rpm' with mean: 5117.59
Filled NaNs in categorical 'num-of-doors' with mode: four

Missing Values after Imputation:
symboling            0
normalized-losses    0
make                 0
fuel-type            0
aspiration           0
num-of-doors         0
body-style           0
drive-wheels         0
engine-location      0
wheel-base           0
length               0
width                0
height               0
curb-weight          0
engine-type          0
num-of-cylinders     0
engine-size          0
fuel-system          0
bore                 0
stroke               0
compression-ratio    0
horsepower           0
peak-rpm             0
city-mpg             0
highway-mpg    

/var/folders/4h/rzfqsskd2bj_xd9b7xfgch_w0000gn/T/ipykernel_64533/2591523121.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(mean_val, inplace=True)
/var/folders/4h/rzfqsskd2bj_xd9b7xfgch_w0000gn/T/ipykernel_64533/2591523121.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [8]:
# --- Imports for Models and Evaluation ---
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression # For regression
# from sklearn.linear_model import LogisticRegression # Use this for classification task
from sklearn.svm import SVR # For regression
# from sklearn.svm import SVC # Use this for classification task
from sklearn.tree import DecisionTreeRegressor # For regression
# from sklearn.tree import DecisionTreeClassifier # Use this for classification task
from sklearn.neural_network import MLPRegressor # For regression
# from sklearn.neural_network import MLPClassifier # Use this for classification task
# from sklearn.metrics import mean_squared_error # Not needed directly with cross_val_score
# from sklearn.metrics import accuracy_score # Use this for classification task
import numpy as np
import pandas as pd # Re-import if in a new script/session

# --- Assume X and y are already defined from Phase 2 ---
# If not, reload and re-preprocess the data as shown in the previous step
# For example:
# file_path = 'datasets/automobile/automobile/imports-85.data'
# column_names = [...] # list all 26 column names
# data = pd.read_csv(file_path, header=None, names=column_names, sep=',', na_values='?')
# ... (rest of the preprocessing steps from Phase 2) ...
# X = data_encoded.drop('price', axis=1)
# y = data_encoded['price']
# numerical_features = X.select_dtypes(include=np.number).columns.tolist() # Needed if pre-processing again

# Make sure X contains only numeric data types before scaling
# Check data types in X
print("\n--- Checking Data Types in X before Pipeline ---")
print(X.info())
# If object types remain, ensure they were handled correctly in Phase 2

# --- Setup for Cross-Validation ---
# 1. Choose K for Cross-Validation
K = 5 # Or K=10, your group decides
kf = KFold(n_splits=K, shuffle=True, random_state=42) # random_state for reproducibility
print(f"\nUsing K={K} for Cross-Validation.")

# --- Define Models for Regression Task ---
# 2. Implement Models (using default parameters initially for Assignment 1)
models_reg = {
    'Linear Regression': LinearRegression(),
    'SVR': SVR(), # Default kernel='rbf', C=1.0
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42), # No depth limit by default
    'MLP Regressor': MLPRegressor(random_state=42,
                                  max_iter=1000, # Increased iterations for convergence
                                  hidden_layer_sizes=(100,), # Default: one hidden layer with 100 neurons
                                  activation='relu', # Default activation
                                  solver='adam', # Default solver
                                  early_stopping=True, # Good practice to prevent overfitting
                                  n_iter_no_change=10) # Stops if validation score doesn't improve
}

# --- Set Task and Metric ---
# For the automobile 'price' prediction, it's regression
current_task = 'regression'
scoring_metric = 'neg_mean_squared_error' # Scikit-learn convention for MSE in CV
metric_name = 'Mean Squared Error (MSE)'
print(f"Task: {current_task}, Evaluation Metric: {metric_name}")


# --- Evaluation Loop using K-Fold CV ---
# 3. Evaluate using K-Fold CV
results = {} # Dictionary to store results

print(f"\n--- Running {K}-Fold Cross-Validation ---")

for model_name, model in models_reg.items():
    # Create a pipeline: Step 1: Scale numerical data, Step 2: Apply model
    # StandardScaler ensures numerical features have zero mean and unit variance
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # Perform K-Fold Cross-Validation using the pipeline
    # This automatically handles:
    # - Splitting data into K folds
    # - Looping K times (using K-1 folds for training, 1 for testing)
    # - Fitting the scaler ONLY on the training part of each fold
    # - Transforming both training and testing parts using the fitted scaler
    # - Fitting the model on the scaled training part
    # - Scoring the model on the scaled testing part
    # - Returning an array of K scores
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring=scoring_metric, n_jobs=-1) # n_jobs=-1 uses all CPU cores

    # Process and store the results
    # cross_val_score returns negative MSE, so we negate the mean
    mean_score = -np.mean(scores)
    std_dev_score = np.std(scores) # Standard deviation of the scores across folds

    results[model_name] = {'Mean MSE': mean_score, 'Std Dev MSE': std_dev_score}

    print(f"{model_name}: Mean MSE = {mean_score:,.2f} (+/- {std_dev_score:,.2f})")

# --- Record Parameters (Example Notes for Report) ---
# 4. Record Parameters
print("\n--- Parameter Notes (Example for Report) ---")
print(f"K-Fold: K={K}, shuffle=True, random_state=42")
print("Scaling: StandardScaler used within pipeline.")
print("Linear Regression: Default parameters.")
print(f"SVR: Default parameters (e.g., kernel='{SVR().kernel}', C={SVR().C}).")
print(f"Decision Tree Regressor: random_state=42, default parameters (e.g., criterion='{DecisionTreeRegressor().criterion}', max_depth=None).")
print(f"MLP Regressor: random_state=42, max_iter=1000, hidden_layer_sizes=(100,), activation='relu', solver='adam', early_stopping=True.")
# Add any specific parameters you changed

# --- Convert results to DataFrame for easy reporting ---
results_df = pd.DataFrame(results).T # Transpose rows/columns
print("\n--- Summary Table for Report ---")
print(results_df.to_markdown()) # Print in Markdown format suitable for report


--- Checking Data Types in X before Pipeline ---
<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, 0 to 204
Data columns (total 64 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   symboling                201 non-null    int64  
 1   normalized-losses        201 non-null    float64
 2   wheel-base               201 non-null    float64
 3   length                   201 non-null    float64
 4   width                    201 non-null    float64
 5   height                   201 non-null    float64
 6   curb-weight              201 non-null    int64  
 7   engine-size              201 non-null    int64  
 8   bore                     201 non-null    float64
 9   stroke                   201 non-null    float64
 10  compression-ratio        201 non-null    float64
 11  horsepower               201 non-null    float64
 12  peak-rpm                 201 non-null    float64
 13  city-mpg                 201 non-nu

/Users/richard/anaconda3/envs/MLE/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
